# Inventory logic: from forecasts to stock decisions

A forecast on its own doesn't tell a store manager what to do. This notebook turns the XGBoost demand
forecasts into inventory decisions for all 500 store-item combinations:
- Which products are at risk of **running out** before a new delivery could arrive?
- **How much** should we reorder?
- Which products have **too much** stock sitting on the shelf?

The result is saved to `data/inventory_status.csv` for the next stage (AI-generated recommendations).

## Business assumptions

These are the rules of the (imaginary) business. They are defined once, here at the top, so you can
change them and re-run the notebook to see how the decisions change.

- **`LEAD_TIME_DAYS`**: how many days it takes for a new order to arrive after we place it.
  Stock has to last at least this long, or the shelf goes empty before the delivery comes.
- **`SAFETY_STOCK_DAYS`**: extra days of stock we keep as a buffer, in case demand is higher than forecast
  or the delivery is late.
- **`FORECAST_DAYS`**: how many days ahead we forecast.
- **`OVERSTOCK_MULTIPLIER`**: we call a product overstocked if its stock is more than this many times
  the demand forecast for the next `FORECAST_DAYS` days.
- **`MIN_STOCK_DAYS` / `MAX_STOCK_DAYS`**: the range used to *simulate* current stock (see Step 3).

In [1]:
LEAD_TIME_DAYS = 7         # days between placing an order and receiving it
SAFETY_STOCK_DAYS = 3      # extra buffer, in days of demand
FORECAST_DAYS = 14         # how far ahead we forecast
OVERSTOCK_MULTIPLIER = 1.5 # stock > 1.5x the next 14 days' demand counts as overstock

MIN_STOCK_DAYS = 5         # simulated stock: at least 5 days' worth of sales...
MAX_STOCK_DAYS = 20        # ...and at most 20 days' worth
RANDOM_SEED = 42           # fixes the "random" stock so every run gives the same numbers

## Step 1: Get a trained model

There were two options: save the model from the all-products notebook to a file and load it here,
or train it again here. **We train it again**, for three reasons:

1. **It's quick.** Training took about 15 seconds in the all-products notebook.
2. **This notebook works on its own.** It doesn't depend on another notebook having been run first
   and a file being in the right place.
3. **It gives a better model for real forecasting.** The earlier model deliberately held back the last
   8 weeks so we could test it. Now we're forecasting the real future, so we train on **all** the data,
   including those most recent 8 weeks. That's the information that matters most for the next 14 days.

The features and model settings are exactly the same as in `forecast_xgboost_all_products.ipynb`.

In [2]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# Load the data and sort so each product's history is one block in date order (same as before)
sales_df = pd.read_csv("../data/train.csv", parse_dates=["date"])
sales_df = sales_df.sort_values(["store", "item", "date"]).reset_index(drop=True)

# Same features as the all-products notebook
sales_df["day_of_week"] = sales_df["date"].dt.dayofweek
sales_df["month"] = sales_df["date"].dt.month
sales_df["year"] = sales_df["date"].dt.year

# groupby + shift/rolling calculate the lags separately for each store-item combination
sales_by_product = sales_df.groupby(["store", "item"])["sales"]
sales_df["lag_7"] = sales_by_product.shift(7)
sales_df["lag_14"] = sales_by_product.shift(14)
sales_df["lag_365"] = sales_by_product.shift(365)
sales_df["rolling_mean_7"] = sales_by_product.transform(lambda group_sales: group_sales.shift(1).rolling(7).mean())

feature_columns = ["day_of_week", "month", "year", "store", "item",
                   "lag_7", "lag_14", "lag_365", "rolling_mean_7"]

# Drop the first year (no lag_365), then train on ALL remaining rows. There's no test split this time.
model_df = sales_df.dropna(subset=feature_columns)

model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, n_jobs=-1, random_state=42)
model.fit(model_df[feature_columns], model_df["sales"])

last_date = sales_df["date"].max()
print(f"Model trained on {len(model_df):,} rows, up to {last_date.date()}.")

Model trained on 730,500 rows, up to 2017-12-31.


## Step 2: Forecast the next 14 days for every product

The data ends on 2017-12-31, so we forecast 2018-01-01 to 2018-01-14.

There's a catch. The model needs `lag_7` (sales 7 days earlier) and `rolling_mean_7` (the average of the
previous 7 days) as inputs. For the first day of the forecast, those days are all in the real data.
For later days they aren't. For example, `lag_7` for 2018-01-10 is the sales on 2018-01-03, which
hasn't happened yet.

The standard fix is to forecast **one day at a time** and use each day's forecast as if it were real
sales when calculating the next day's features:

1. Forecast 2018-01-01 using real history.
2. Add those forecasts to the history.
3. Forecast 2018-01-02, whose `rolling_mean_7` now includes the 2018-01-01 forecast.
4. Repeat for all 14 days.

This is called a **recursive forecast**. Errors can build up a little over the 14 days, because later days
are partly based on earlier forecasts rather than real sales.

In [3]:
# .pivot() reshapes the long table into a wide one: one row per date, one column per store-item.
# This makes "sales on date X for every product" a single row, which is easy to look up.
#
#   date        (1,1)  (1,2)  ...  (10,50)
#   2017-12-30   19     52          71
#   2017-12-31   23     49          68
sales_history = sales_df.pivot(index="date", columns=["store", "item"], values="sales").astype(float)

# pd.date_range() makes a list of consecutive dates: the 14 days after the data ends
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq="D")

# The list of all 500 (store, item) pairs, taken from the wide table's column names
products = sales_history.columns

for forecast_date in forecast_dates:
    # .loc[date] picks the row for that date: one value per product.
    # Earlier forecasts are already added to sales_history, so these look-ups work for every day.
    features_for_day = pd.DataFrame({
        "day_of_week": forecast_date.dayofweek,
        "month": forecast_date.month,
        "year": forecast_date.year,
        "store": products.get_level_values("store"),
        "item": products.get_level_values("item"),
        "lag_7": sales_history.loc[forecast_date - pd.Timedelta(days=7)].to_numpy(),
        "lag_14": sales_history.loc[forecast_date - pd.Timedelta(days=14)].to_numpy(),
        "lag_365": sales_history.loc[forecast_date - pd.Timedelta(days=365)].to_numpy(),
        # .iloc[-7:] takes the last 7 rows of the history (the 7 days before forecast_date);
        # .mean() averages each product's column
        "rolling_mean_7": sales_history.iloc[-7:].mean().to_numpy(),
    })

    # Predict all 500 products for this day. Sales can't be negative, so clip at 0.
    predictions = model.predict(features_for_day[feature_columns]).clip(min=0)

    # Add the forecast as a new row at the bottom of the history, so the next day can use it
    sales_history.loc[forecast_date] = predictions

# Keep only the 14 forecast rows, then turn the wide table back into a long one:
# .rename_axis("date") names the row labels "date" (adding rows above dropped that name),
# .stack() moves the store and item column labels back into rows, and .reset_index()
# turns them all into ordinary columns, giving one row per (date, store, item)
forecast_df = (
    sales_history.loc[forecast_dates]
    .rename_axis("date")
    .stack(["store", "item"])
    .rename("predicted_sales")
    .reset_index()
)

print(f"Forecast {len(forecast_df):,} rows: {FORECAST_DAYS} days x {len(products)} products "
      f"({forecast_dates[0].date()} to {forecast_dates[-1].date()})")
forecast_df.head()

Forecast 7,000 rows: 14 days x 500 products (2018-01-01 to 2018-01-14)


,date,store,item,predicted_sales
0,2018-01-01,1,1,14.464958
1,2018-01-01,1,2,37.077347
2,2018-01-01,1,3,20.732018
3,2018-01-01,1,4,13.183475
4,2018-01-01,1,5,11.504566


## Step 3: Simulate the current stock (an assumption)

> **Important: this is simulated data.** The dataset contains sales only. It has **no real inventory
> numbers**, so we make up a plausible current stock level for each product purely for demonstration.
> In a real system this column would come from the store's inventory database. Every stock-related result
> below (stockout risk, reorder quantities, overstock flags) is therefore based on this assumption,
> **not** on real stock.

How we simulate it: each product gets a random number of **days' worth of stock**, between 5 and 20,
multiplied by that product's **average daily sales over the last 28 days** of real data.
For example, a product selling 30 units a day that draws 10 days would get 300 units of stock.
We base it on each product's own sales so the numbers are realistic: a fast-selling item gets a lot of
stock, a slow one gets a little.

In [4]:
# Average real daily sales over the last 28 days, for each product.
# .tail(28) takes the last 28 rows of each group (the data is sorted by date within each group).
recent_sales = sales_df.groupby(["store", "item"]).tail(28)
inventory_df = (
    recent_sales.groupby(["store", "item"])["sales"].mean()
    .rename("avg_daily_sales_recent")
    .reset_index()
)

# np.random.default_rng(seed) creates a random number generator. The seed makes the "random"
# numbers the same every run; change RANDOM_SEED to get a different simulated stock.
# .uniform(low, high, size) draws one random decimal number per product in that range.
rng = np.random.default_rng(RANDOM_SEED)
inventory_df["simulated_stock_days"] = rng.uniform(MIN_STOCK_DAYS, MAX_STOCK_DAYS, size=len(inventory_df))

# Stock in units = days of stock x average daily sales, rounded to whole units
inventory_df["current_stock"] = (
    inventory_df["simulated_stock_days"] * inventory_df["avg_daily_sales_recent"]
).round().astype(int)

inventory_df.head()

,store,item,avg_daily_sales_recent,simulated_stock_days,current_stock
0,1,1,16.678571,16.609341,277
1,1,2,46.285714,11.583177,536
2,1,3,27.785714,17.878969,497
3,1,4,15.821429,15.460520,245
4,1,5,13.785714,6.412660,88


## Step 4: Calculate the inventory metrics

For each product we work out:

| Column | Meaning |
|---|---|
| `avg_daily_predicted_demand` | average forecast sales per day over the next 14 days |
| `predicted_demand_leadtime` | total forecast sales during the next `LEAD_TIME_DAYS` (7) days, i.e. what we'll sell while waiting for a delivery |
| `days_of_stock_left` | how many days the current stock lasts at the forecast rate |
| `stockout_risk` | **True** if stock runs out *before* a new order could arrive (`days_of_stock_left < LEAD_TIME_DAYS`) |
| `reorder_qty` | how many units to order now: enough to cover demand during the lead time **plus** a safety buffer of `SAFETY_STOCK_DAYS` days, minus what we already have. Never below 0. |
| `overstock_flag` | **True** if stock is more than 1.5x the whole 14-day forecast, i.e. money tied up in stock that won't sell soon |

In [5]:
# --- Totals and averages from the 14-day forecast ---
# .groupby(...).agg(new_name=(column, function)) calculates several summaries at once, one row per product
forecast_summary = forecast_df.groupby(["store", "item"]).agg(
    avg_daily_predicted_demand=("predicted_sales", "mean"),
    predicted_demand_14d=("predicted_sales", "sum"),
).reset_index()

# Demand during the lead time: only the first LEAD_TIME_DAYS forecast days.
# .isin() checks whether each date is in the given list of dates.
lead_time_dates = forecast_dates[:LEAD_TIME_DAYS]
lead_time_demand = (
    forecast_df[forecast_df["date"].isin(lead_time_dates)]
    .groupby(["store", "item"])["predicted_sales"].sum()
    .rename("predicted_demand_leadtime")
    .reset_index()
)

# .merge() joins tables side by side, matching rows on the store and item columns (like VLOOKUP in Excel)
inventory_df = inventory_df.merge(forecast_summary, on=["store", "item"]).merge(lead_time_demand, on=["store", "item"])

# --- Days of stock left ---
inventory_df["days_of_stock_left"] = inventory_df["current_stock"] / inventory_df["avg_daily_predicted_demand"]

# --- Stockout risk: will we run out before a new order arrives? ---
inventory_df["stockout_risk"] = inventory_df["days_of_stock_left"] < LEAD_TIME_DAYS

# --- Reorder quantity ---
# Safety stock in units = SAFETY_STOCK_DAYS days of forecast demand
safety_stock_units = SAFETY_STOCK_DAYS * inventory_df["avg_daily_predicted_demand"]
units_needed = inventory_df["predicted_demand_leadtime"] + safety_stock_units - inventory_df["current_stock"]

# .clip(lower=0) turns any negative number into 0: if we already have enough, order nothing.
# np.ceil() rounds UP to whole units, because we can't order part of an item and rounding down
# would leave us slightly short.
inventory_df["reorder_qty"] = np.ceil(units_needed.clip(lower=0)).astype(int)

# --- Overstock: far more stock than the next 14 days need ---
inventory_df["overstock_flag"] = inventory_df["current_stock"] > inventory_df["predicted_demand_14d"] * OVERSTOCK_MULTIPLIER

inventory_df.head()

,store,item,avg_daily_sales_recent,simulated_stock_days,current_stock,avg_daily_predicted_demand,predicted_demand_14d,predicted_demand_leadtime,days_of_stock_left,stockout_risk,reorder_qty,overstock_flag
0,1,1,16.678571,16.609341,277,17.210586,240.948202,122.816589,16.094746,False,0,False
1,1,2,46.285714,11.583177,536,46.281406,647.939686,328.768024,11.581325,False,0,False
2,1,3,27.785714,17.878969,497,25.496949,356.957293,178.560518,19.492528,False,0,False
3,1,4,15.821429,15.460520,245,16.082318,225.152452,114.807308,15.234122,False,0,False
4,1,5,13.785714,6.412660,88,14.292056,200.088785,101.885232,6.157267,True,57,False


## Step 5: Build the summary table, most urgent first

We keep only the columns a store manager needs and sort by `days_of_stock_left` from smallest to largest,
so the products closest to running out are at the top.

In [6]:
summary_columns = ["store", "item", "current_stock", "avg_daily_predicted_demand",
                   "days_of_stock_left", "stockout_risk", "reorder_qty", "overstock_flag"]

inventory_status = (
    inventory_df[summary_columns]
    .sort_values("days_of_stock_left")  # ascending by default: smallest (most urgent) first
    .reset_index(drop=True)
    .round({"avg_daily_predicted_demand": 1, "days_of_stock_left": 1})  # round only these columns
)

inventory_status.head(15)

,store,item,current_stock,avg_daily_predicted_demand,days_of_stock_left,stockout_risk,reorder_qty,overstock_flag
0,5,39,141,27.7,5.1,True,139,False
1,9,23,132,25.9,5.1,True,127,False
2,5,27,75,14.0,5.4,True,66,False
3,3,36,393,72.6,5.4,True,340,False
4,6,20,169,31.2,5.4,True,145,False
5,2,2,325,59.8,5.4,True,275,False
6,6,9,205,37.0,5.5,True,166,False
7,8,45,423,75.8,5.6,True,342,False
8,9,35,333,59.7,5.6,True,269,False
9,6,21,147,26.1,5.6,True,116,False


## Step 6: How many products need attention?

In [7]:
# Summing a True/False column counts the Trues (True counts as 1, False as 0)
num_products = len(inventory_status)
num_stockout_risk = inventory_status["stockout_risk"].sum()
num_overstock = inventory_status["overstock_flag"].sum()
num_to_reorder = (inventory_status["reorder_qty"] > 0).sum()

print(f"Stockout risk (runs out within {LEAD_TIME_DAYS} days): {num_stockout_risk} of {num_products} products")
print(f"Overstocked (> {OVERSTOCK_MULTIPLIER}x the next {FORECAST_DAYS} days' demand): {num_overstock} of {num_products} products")
print(f"Need a reorder now (reorder_qty > 0): {num_to_reorder} of {num_products} products")
print("\nReminder: current stock is simulated, so these counts show how the logic works, not real store status.")

Stockout risk (runs out within 7 days): 61 of 500 products
Overstocked (> 1.5x the next 14 days' demand): 11 of 500 products
Need a reorder now (reorder_qty > 0): 163 of 500 products

Reminder: current stock is simulated, so these counts show how the logic works, not real store status.


## Step 7: Save the results for the next stages

We save two CSV files:
- **`inventory_status.csv`**: the summary table. The AI recommendations notebook reads it.
- **`demand_forecast_14day.csv`**: the day-by-day forecast from Step 2 (one row per store, item and date).
  The dashboard reads it to draw each product's forecast. Saving it here means the dashboard doesn't
  have to re-run the model, which takes about 40 seconds, every time it starts.

In [8]:
output_path = "../data/inventory_status.csv"

# .to_csv() writes the table to a file; index=False leaves out pandas' row numbers
inventory_status.to_csv(output_path, index=False)
print(f"Saved {len(inventory_status)} rows to {output_path}")

forecast_path = "../data/demand_forecast_14day.csv"
forecast_output = forecast_df[["store", "item", "date", "predicted_sales"]].round({"predicted_sales": 2})
forecast_output.to_csv(forecast_path, index=False)
print(f"Saved {len(forecast_output)} rows to {forecast_path}")

Saved 500 rows to ../data/inventory_status.csv
Saved 7000 rows to ../data/demand_forecast_14day.csv
